In [16]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_claims = 35000

#=========================
# Catálogos
#=========================

branches = {
    "Auto":{
        "coverage":["Third Party","Full Coverage"],
        "claim_types":["Collision","Theft","Fire","Glass"],
        "premium":(80000,250000),
        "sum_insured":(4000000,20000000),
        "paid":(50000,3000000)
    },

    "ART":{
        "coverage":["Workers Compensation"],
        "claim_types":["Temporary Disability","Permanent Disability","Death"],
        "premium":(500000,3000000),
        "sum_insured":(20000000,100000000),
        "paid":(500000,10000000)
    },

    "Vida":{
        "coverage":["Individual","Collective"],
        "claim_types":["Death","Disability"],
        "premium":(50000,500000),
        "sum_insured":(5000000,100000000),
        "paid":(1000000,50000000)
    },

    "Hogar":{
        "coverage":["Basic","Premium"],
        "claim_types":["Fire","Theft","Water Damage"],
        "premium":(40000,180000),
        "sum_insured":(3000000,30000000),
        "paid":(30000,2000000)
    },

    "Comercio":{
        "coverage":["Integral"],
        "claim_types":["Fire","Liability","Theft"],
        "premium":(80000,600000),
        "sum_insured":(5000000,50000000),
        "paid":(50000,5000000)
    }
}

branch_prob=[0.45,0.20,0.10,0.15,0.10]

provinces=[
"Buenos Aires","CABA","Córdoba","Santa Fe",
"Mendoza","Tucumán","Salta","Neuquén"
]

genders=["Male","Female"]

status=["Open","Closed"]

#=========================
# Fechas
#=========================

accident_dates=pd.to_datetime(
    np.random.choice(
        pd.date_range("2015-01-01","2024-12-31"),
        n_claims
    )
)

report_delay=np.random.randint(0,45,n_claims)

report_dates=accident_dates+pd.to_timedelta(report_delay,unit="D")

payment_delay=np.random.randint(5,540,n_claims)

payment_dates=report_dates+pd.to_timedelta(payment_delay,unit="D")

#=========================
# Generación
#=========================

rows=[]

selected_branches=np.random.choice(
    list(branches.keys()),
    n_claims,
    p=branch_prob
)

for i,branch in enumerate(selected_branches):

    info=branches[branch]

    coverage=np.random.choice(info["coverage"])

    claim_type=np.random.choice(info["claim_types"])

    premium=np.random.randint(*info["premium"])

    sum_insured=np.random.randint(*info["sum_insured"])

    paid=np.random.randint(*info["paid"])

    st=np.random.choice(status,p=[0.25,0.75])

    reserve=0 if st=="Closed" else np.random.randint(
        int(paid*0.05),
        int(paid*0.80)
    )

    incurred=paid+reserve

    rows.append({

        "Claim_ID":f"CLM{i+1:07}",

        "Policy_ID":f"POL{np.random.randint(1,9000):06}",

        "Customer_ID":f"CUS{np.random.randint(1,15000):06}",

        "Branch":branch,

        "Coverage":coverage,

        "Claim_Type":claim_type,

        "Province":np.random.choice(provinces),

        "Policy_Start":accident_dates[i]-pd.Timedelta(days=np.random.randint(30,1200)),

        "Policy_End":accident_dates[i]+pd.Timedelta(days=365),

        "Accident_Date":accident_dates[i],

        "Report_Date":report_dates[i],

        "Payment_Date":payment_dates[i],

        "Accident_Year":accident_dates[i].year,

        "Accident_Month":accident_dates[i].month,

        "Report_Delay_Days":report_delay[i],

        "Development_Month":((payment_dates[i]-accident_dates[i]).days//30)+1,

        "Premium":premium,

        "Sum_Insured":sum_insured,

        "Paid_Amount":paid,

        "Outstanding_Reserve":reserve,

        "Incurred":incurred,

        "Claim_Count":1,

        "Status":st,

        "Customer_Age":np.random.randint(18,85),

        "Customer_Gender":np.random.choice(genders),

        "Vehicle_Age":np.random.randint(0,25) if branch=="Auto" else np.nan,

        "Fraud_Flag":np.random.choice([0,1],p=[0.98,0.02])

    })

df_claims=pd.DataFrame(rows)

In [18]:
df_claims.head()

,Claim_ID,Policy_ID,Customer_ID,Branch,Coverage,Claim_Type,Province,Policy_Start,Policy_End,Accident_Date,...,Sum_Insured,Paid_Amount,Outstanding_Reserve,Incurred,Claim_Count,Status,Customer_Age,Customer_Gender,Vehicle_Age,Fraud_Flag
0,CLM0000001,POL001143,CUS000794,Comercio,Integral,Theft,Córdoba,2023-02-13,2024-09-09,2023-09-10,...,34833375,4614697,1288424,5903121,1,Open,81,Female,NaN,0
1,CLM0000002,POL005361,CUS000703,Comercio,Integral,Fire,Neuquén,2023-11-06,2025-08-08,2024-08-08,...,11646516,295524,32751,328275,1,Open,27,Male,NaN,0
2,CLM0000003,POL003207,CUS002613,Vida,Individual,Death,Buenos Aires,2016-07-22,2018-05-10,2017-05-10,...,14690589,36224539,0,36224539,1,Closed,62,Male,NaN,1
3,CLM0000004,POL001826,CUS010240,Hogar,Premium,Water Damage,Buenos Aires,2016-03-12,2019-07-18,2018-07-18,...,15466625,1241065,0,1241065,1,Closed,48,Male,NaN,0
4,CLM0000005,POL003772,CUS010277,Auto,Third Party,Theft,Neuquén,2015-05-11,2019-02-04,2018-02-04,...,15008264,452276,0,452276,1,Closed,50,Female,23.0,0


In [19]:
df_claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Claim_ID             35000 non-null  object        
 1   Policy_ID            35000 non-null  object        
 2   Customer_ID          35000 non-null  object        
 3   Branch               35000 non-null  object        
 4   Coverage             35000 non-null  object        
 5   Claim_Type           35000 non-null  object        
 6   Province             35000 non-null  object        
 7   Policy_Start         35000 non-null  datetime64[ns]
 8   Policy_End           35000 non-null  datetime64[ns]
 9   Accident_Date        35000 non-null  datetime64[ns]
 10  Report_Date          35000 non-null  datetime64[ns]
 11  Payment_Date         35000 non-null  datetime64[ns]
 12  Accident_Year        35000 non-null  int64         
 13  Accident_Month       35000 non-

In [20]:
df_claims.describe(include="all")

,Claim_ID,Policy_ID,Customer_ID,Branch,Coverage,Claim_Type,Province,Policy_Start,Policy_End,Accident_Date,...,Sum_Insured,Paid_Amount,Outstanding_Reserve,Incurred,Claim_Count,Status,Customer_Age,Customer_Gender,Vehicle_Age,Fraud_Flag
count,35000,35000,35000,35000,35000,35000,35000,35000,35000,35000,...,3.500000e+04,3.500000e+04,3.500000e+04,3.500000e+04,35000.0,35000,35000.000000,35000,15671.000000,35000.000000
unique,35000,8834,13516,5,8,10,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2,NaN,2,NaN,NaN
top,CLM0035000,POL000579,CUS014039,Auto,Third Party,Fire,Córdoba,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Closed,NaN,Female,NaN,NaN
freq,1,14,10,15671,7841,6909,4504,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,26217,NaN,17565,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-30 02:33:52.457142784,2021-01-04 19:52:43.885714176,2020-01-05 19:52:43.885714432,...,2.786762e+07,4.680963e+06,4.944010e+05,5.175364e+06,1.0,NaN,50.825029,NaN,12.012124,0.020429
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-10-06 00:00:00,2016-01-01 00:00:00,2015-01-01 00:00:00,...,3.000119e+06,3.008400e+04,0.000000e+00,3.008400e+04,1.0,NaN,18.000000,NaN,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-11-06 18:00:00,2018-07-14 00:00:00,2017-07-14 00:00:00,...,1.071755e+07,1.005577e+06,0.000000e+00,1.077575e+06,1.0,NaN,34.000000,NaN,6.000000,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-20 00:00:00,2020-12-28 00:00:00,2019-12-29 00:00:00,...,1.751422e+07,1.929914e+06,0.000000e+00,2.092660e+06,1.0,NaN,51.000000,NaN,12.000000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-11-03 00:00:00,2023-07-12 00:00:00,2022-07-12 00:00:00,...,3.739188e+07,3.793202e+06,1.145100e+04,4.232681e+06,1.0,NaN,68.000000,NaN,18.000000,0.000000
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-05 00:00:00,2025-12-31 00:00:00,2024-12-31 00:00:00,...,9.999459e+07,4.999106e+07,3.838921e+07,8.780567e+07,1.0,NaN,84.000000,NaN,24.000000,1.000000


In [21]:
df_claims.isna().sum()

,0
Claim_ID,0
Policy_ID,0
Customer_ID,0
Branch,0
Coverage,0
Claim_Type,0
Province,0
Policy_Start,0
Policy_End,0
Accident_Date,0


In [22]:
df_claims["Branch"].value_counts()

,count
Branch,
Auto,15671
ART,7083
Hogar,5346
Vida,3479
Comercio,3421


In [23]:
df_claims.groupby("Branch")["Paid_Amount"].describe()

,count,mean,std,min,25%,50%,75%,max
Branch,,,,,,,,
ART,7083.0,5.194528e+06,2.722255e+06,500046.0,2848621.00,5165896.0,7569458.00,9997659.0
Auto,15671.0,1.522773e+06,8.534383e+05,50025.0,786629.00,1528350.0,2265504.00,2999754.0
Comercio,3421.0,2.539697e+06,1.430419e+06,50656.0,1302922.00,2530898.0,3793554.00,4999361.0
Hogar,5346.0,1.019861e+06,5.729350e+05,30084.0,520596.25,1023739.5,1511284.25,1998280.0
Vida,3479.0,2.559271e+07,1.431202e+07,1000605.0,13115083.50,25363850.0,38183356.00,49991061.0


In [24]:
df_claims["Development_Month"].describe()

,Development_Month
count,35000.000000
mean,10.315829
std,5.162722
min,1.000000
25%,6.000000
50%,10.000000
75%,15.000000
max,20.000000


In [25]:
# Data Quality Check

print(df_claims.isna().sum())

Claim_ID                   0
Policy_ID                  0
Customer_ID                0
Branch                     0
Coverage                   0
Claim_Type                 0
Province                   0
Policy_Start               0
Policy_End                 0
Accident_Date              0
Report_Date                0
Payment_Date               0
Accident_Year              0
Accident_Month             0
Report_Delay_Days          0
Development_Month          0
Premium                    0
Sum_Insured                0
Paid_Amount                0
Outstanding_Reserve        0
Incurred                   0
Claim_Count                0
Status                     0
Customer_Age               0
Customer_Gender            0
Vehicle_Age            19329
Fraud_Flag                 0
dtype: int64


In [26]:
# Vehicle_Age only applies to Auto policies

df_claims[df_claims["Branch"]!="Auto"]["Vehicle_Age"].isna().all()

np.True_

In [30]:
df_claims.to_excel(
    "claims_dataset_v1.xlsx",
    index=False
)

In [32]:
from google.colab import files

df_claims.to_excel("claims_dataset_v1.xlsx", index=False)

files.download("claims_dataset_v1.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>